# Eficácia do Pressing no Futebol de Elite — Análise Quantitativa

## 1. Introdução e Objetivo

Este notebook implementa a **metodologia inicial** da proposta de projeto (`Pressao_no_futebol_v1.pdf`), aplicada à **UEFA Euro 2020** (StatsBomb Open Data — dados de evento + freeze frames 360).

A unidade de análise é o **evento individual de pressão** (`type_name == Pressure`) e a variável resposta é `recovered`: o time pressionante recuperou a bola em até 10 segundos.

**Foco desta etapa:** avaliar o *input* do modelo. Cada feature candidata passa por **testes de hipótese formais** — qui-quadrado, Mann-Whitney, Wald, razão de verossimilhança (LRT) e permutação — para justificar empiricamente sua inclusão, conforme a Seção A.5 do PDF.

A profundidade dos testes é controlada pela flag `TEST_LEVEL` na célula de configuração. Os resultados por variável são persistidos em `variable_analysis/` (Seção 5.6).

## 2. Configuração e Carregamento de Dados

In [ ]:
import json
import os
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from shapely.geometry import Polygon, MultiPoint, Point
from shapely.ops import voronoi_diagram
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print('Bibliotecas carregadas.')

In [ ]:
# Nivel de testes de hipotese executados na Secao 5:
#   1 = triagem univariada (qui-quadrado / Mann-Whitney)             [padrao]
#   2 = + VIF + modelo logistico (Wald) + razao de verossimilhanca (LRT)
#   3 = + teste de permutacao
TEST_LEVEL = 1

DATA_DIR = Path('StatsBomb_2/data')
if not DATA_DIR.exists():
    DATA_DIR = Path('C:/Users/Abraao Ideapad3/Documents/Projetos/COE609/Analise_pressao_1/StatsBomb_2/data')

COMP_ID, SEASON_ID = 55, 43       # UEFA Euro 2020
FIELD_X, FIELD_Y = 120.0, 80.0    # campo StatsBomb
RECOVERY_WINDOW = 10.0            # janela de recuperacao (segundos)
ALPHA = 0.05                      # nivel de significancia
VAR_DIR = 'variable_analysis'     # pasta de saida das tabelas por variavel

print('TEST_LEVEL =', TEST_LEVEL)
print('Diretorio de dados:', DATA_DIR, '| existe:', DATA_DIR.exists())

In [ ]:
def load_json(path):
    with open(path, encoding='utf-8') as fh:
        return json.load(fh)

matches = load_json(DATA_DIR / 'matches' / str(COMP_ID) / (str(SEASON_ID) + '.json'))
match_ids = sorted(m['match_id'] for m in matches)
print(len(match_ids), 'jogos na UEFA Euro 2020')

events_parts = []
frames = {}
for mid in match_ids:
    raw = load_json(DATA_DIR / 'events' / (str(mid) + '.json'))
    edf = pd.json_normalize(raw, sep='_')
    edf['match_id'] = mid
    events_parts.append(edf)
    ts_path = DATA_DIR / 'three-sixty' / (str(mid) + '.json')
    if ts_path.exists():
        for fr in load_json(ts_path):
            frames[fr['event_uuid']] = fr

events = pd.concat(events_parts, ignore_index=True)
print('Total de eventos:', len(events))
print('Freeze frames 360 carregados:', len(frames))

## 3. Unidade de Análise e Variável Resposta

A unidade de análise é o evento de pressão. A resposta `recovered` é construída por **junção temporal** (`merge_asof`): para cada pressão, procura-se o próximo evento de recuperação (`Ball Recovery`, `Interception` ou `Duel` vencido) realizado pelo **mesmo time**, na mesma partida e período, dentro de 10 segundos.

In [ ]:
def ts_to_seconds(ts):
    hh, mm, ss = ts.split(':')
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

events['t'] = events['timestamp'].apply(ts_to_seconds)
events['t_match'] = events['minute'] * 60 + events['second']

pressures = events[events['type_name'] == 'Pressure'].copy()
print('Eventos de pressao:', len(pressures))

# Eventos de recuperacao de bola
if 'ball_recovery_recovery_failure' in events.columns:
    rec_fail = events['ball_recovery_recovery_failure'].fillna(False)
else:
    rec_fail = pd.Series(False, index=events.index)

duel_outcome = events.get('duel_outcome_name', pd.Series('', index=events.index)).fillna('')
duel_won = events['type_name'].eq('Duel') & duel_outcome.str.contains('Won|Success')

rec_mask = (
    (events['type_name'].eq('Ball Recovery') & ~rec_fail)
    | events['type_name'].eq('Interception')
    | duel_won
)
recoveries = events.loc[rec_mask, ['match_id', 'period', 'team_id', 't']].copy()
print('Eventos de recuperacao:', len(recoveries))

left = (pressures[['id', 'match_id', 'period', 'team_id', 't']]
        .rename(columns={'t': 't_press'}).sort_values('t_press'))
right = recoveries.rename(columns={'t': 't_rec'}).sort_values('t_rec')

matched = pd.merge_asof(
    left, right,
    left_on='t_press', right_on='t_rec',
    by=['match_id', 'period', 'team_id'],
    direction='forward', tolerance=RECOVERY_WINDOW)
matched['recovered'] = matched['t_rec'].notna().astype(int)

pressures = pressures.merge(matched[['id', 'recovered']], on='id', how='left')
pressures['recovered'] = pressures['recovered'].fillna(0).astype(int)
print('Taxa de recuperacao observada: %.1f%% (PDF: ~25.5%%)' % (100 * pressures['recovered'].mean()))

## 4. Engenharia de Features

### 4.1 Features posicionais (PDF 2.4.2)

- `dist_to_goal`: distância euclidiana ao gol adversário (120, 40).
- `zone`: terço do campo (Defensivo x≤40, Médio 40–80, Atacante >80).
- `duration`: duração do evento de pressão (atributo direto).

In [ ]:
xy = pressures['location'].apply(
    lambda v: v if isinstance(v, list) and len(v) == 2 else [np.nan, np.nan])
pressures['x'] = xy.apply(lambda v: v[0])
pressures['y'] = xy.apply(lambda v: v[1])

pressures['dist_to_goal'] = np.sqrt((FIELD_X - pressures['x']) ** 2 + (40.0 - pressures['y']) ** 2)
pressures['zone'] = pd.cut(pressures['x'], bins=[-0.1, 40, 80, 120.1],
                           labels=['Defensivo', 'Medio', 'Atacante']).astype(str)
pressures['duration'] = pressures['duration'].fillna(0.0)
print(pressures[['x', 'y', 'dist_to_goal', 'duration']].describe())
print(pressures['zone'].value_counts())

### 4.2 Features de contexto de jogo (PDF 2.4.3)

- `score_diff`: placar corrido (gols do time pressionante − gols do adversário) até o instante da pressão. Gols vêm de `Shot` com `outcome == Goal` e de eventos `Own Goal For`.
- `minute`: minuto da partida (atributo direto).

In [ ]:
shot_outcome = events.get('shot_outcome_name', pd.Series('', index=events.index))
goals = events[
    (events['type_name'].eq('Shot') & shot_outcome.eq('Goal'))
    | events['type_name'].eq('Own Goal For')
][['match_id', 'team_id', 't_match']].copy()
print('Gols identificados:', len(goals))

score_diff = pd.Series(0, index=pressures.index, dtype=int)
for mid, grp in pressures.groupby('match_id'):
    g = goals[goals['match_id'] == mid]
    g_t = g['t_match'].to_numpy()
    g_team = g['team_id'].to_numpy()
    for idx, team, tm in zip(grp.index, grp['team_id'].to_numpy(), grp['t_match'].to_numpy()):
        before = g_t <= tm
        own = int(((g_team == team) & before).sum())
        opp = int(((g_team != team) & before).sum())
        score_diff.at[idx] = own - opp

pressures['score_diff'] = score_diff
print(pressures['score_diff'].value_counts().sort_index())

### 4.3 Features táticas (PDF 2.4.4)

O time pressionado é o `possession_team` da pressão.

- `trigger_type`: tipo do evento mais recente do time pressionado antes da pressão (`merge_asof` backward, 5 s).
- `action_under_pressure`: evento do time pressionado sob pressão mais próximo (`merge_asof` nearest, 2 s).
- `is_counterpress`: a pressão ocorreu ≤5 s após perda de bola do próprio time (`Dispossessed`, `Miscontrol`, `Pass` incompleto/fora). Comparado com o atributo nativo `counterpress`.

In [ ]:
ev_all = events[['match_id', 'period', 't', 'team_id', 'type_name', 'under_pressure']].copy()

# trigger_type: ultimo evento do time pressionado antes da pressao
trig_left = (pressures[['id', 'match_id', 'period', 't', 'possession_team_id']]
             .rename(columns={'possession_team_id': 'side'}).sort_values('t'))
trig_right = (ev_all.rename(columns={'team_id': 'side', 'type_name': 'trigger_type'})
              [['match_id', 'period', 't', 'side', 'trigger_type']].sort_values('t'))
trig = pd.merge_asof(trig_left, trig_right, on='t',
                     by=['match_id', 'period', 'side'],
                     direction='backward', tolerance=5.0)
pressures = pressures.merge(trig[['id', 'trigger_type']], on='id', how='left')

# action_under_pressure: evento do time pressionado sob pressao
aup_right = (ev_all[ev_all['under_pressure'] == True]
             .rename(columns={'team_id': 'side', 'type_name': 'action_under_pressure'})
             [['match_id', 'period', 't', 'side', 'action_under_pressure']].sort_values('t'))
aup = pd.merge_asof(trig_left, aup_right, on='t',
                    by=['match_id', 'period', 'side'],
                    direction='nearest', tolerance=2.0)
pressures = pressures.merge(aup[['id', 'action_under_pressure']], on='id', how='left')

# is_counterpress: perda do proprio time ate 5s antes
pass_outcome = events.get('pass_outcome_name', pd.Series('', index=events.index))
loss_mask = (events['type_name'].isin(['Dispossessed', 'Miscontrol'])
             | (events['type_name'].eq('Pass') & pass_outcome.isin(['Incomplete', 'Out'])))
loss = events.loc[loss_mask, ['match_id', 'period', 't', 'team_id']].rename(columns={'team_id': 'side'}).copy()
loss['lost'] = 1
loss = loss.sort_values('t')
cp_left = (pressures[['id', 'match_id', 'period', 't', 'team_id']]
           .rename(columns={'team_id': 'side'}).sort_values('t'))
cp = pd.merge_asof(cp_left, loss, on='t', by=['match_id', 'period', 'side'],
                   direction='backward', tolerance=5.0)
cp['is_counterpress'] = cp['lost'].notna().astype(int)
pressures = pressures.merge(cp[['id', 'is_counterpress']], on='id', how='left')

print('trigger_type:')
print(pressures['trigger_type'].value_counts(dropna=False).head(8))
print('action_under_pressure:')
print(pressures['action_under_pressure'].value_counts(dropna=False).head(8))
print('is_counterpress (taxa): %.1f%%' % (100 * pressures['is_counterpress'].mean()))
if 'counterpress' in pressures.columns:
    nativo = pressures['counterpress'].fillna(False).astype(bool).mean()
    print('counterpress nativo StatsBomb (taxa): %.1f%%' % (100 * nativo))

### 4.4 Ações defensivas concorrentes (PDF 2.4.5)

`had_tackle`, `had_block`, `had_foul`: houve desarme / bloqueio / falta cometida pelo time pressionante em até 3 s após a pressão (`merge_asof` forward).

In [ ]:
def had_within(press_df, subset, tol=3.0):
    right = subset[['match_id', 'period', 't', 'team_id']].rename(columns={'team_id': 'side'}).copy()
    right['flag'] = 1
    right = right.sort_values('t')
    lft = (press_df[['id', 'match_id', 'period', 't', 'team_id']]
           .rename(columns={'team_id': 'side'}).sort_values('t'))
    m = pd.merge_asof(lft, right, on='t', by=['match_id', 'period', 'side'],
                      direction='forward', tolerance=tol)
    return m.set_index('id')['flag'].notna().astype(int)

duel_type = events.get('duel_type_name', pd.Series('', index=events.index))
tackles = events[events['type_name'].eq('Duel') & duel_type.eq('Tackle')]
blocks = events[events['type_name'].eq('Block')]
fouls = events[events['type_name'].eq('Foul Committed')]

pressures = pressures.set_index('id')
for name, subset in [('had_tackle', tackles), ('had_block', blocks), ('had_foul', fouls)]:
    pressures[name] = had_within(pressures.reset_index(), subset)
pressures = pressures.reset_index()
print(pressures[['had_tackle', 'had_block', 'had_foul']].mean())

### 4.5 Features 360 — freeze frame (PDF 2.4.1)

Para cada pressão, o freeze frame é localizado por `event_uuid`. A posição da bola é aproximada pela `location` do evento de pressão. Companheiros são os jogadores com `teammate == True` (time pressionante).

*Caveat:* o freeze frame 360 contém apenas jogadores **visíveis** no instante do evento.

In [ ]:
F360_KEYS = ['n_teammates_10m', 'n_adversaries_5m', 'n_opponents_10m',
             'nearest_teammate_dist', 'nearest_opponent_dist',
             'numerical_superiority', 'pressing_compactness']

def frame_360_features(row):
    fr = frames.get(row['id'])
    if fr is None or pd.isna(row['x']):
        return pd.Series({k: np.nan for k in F360_KEYS})
    bx, by = row['x'], row['y']
    mate_d, opp_d = [], []
    for p in fr['freeze_frame']:
        loc = p['location']
        d = float(np.hypot(loc[0] - bx, loc[1] - by))
        if p.get('teammate'):
            mate_d.append(d)
        else:
            opp_d.append(d)
    mate_d = np.array(mate_d)
    opp_d = np.array(opp_d)
    n_tm10 = int((mate_d <= 10).sum())
    n_op10 = int((opp_d <= 10).sum())
    return pd.Series({
        'n_teammates_10m': n_tm10,
        'n_adversaries_5m': int((opp_d <= 5).sum()),
        'n_opponents_10m': n_op10,
        'nearest_teammate_dist': float(mate_d.min()) if len(mate_d) else np.nan,
        'nearest_opponent_dist': float(opp_d.min()) if len(opp_d) else np.nan,
        'numerical_superiority': n_tm10 - n_op10,
        'pressing_compactness': float(mate_d.std()) if len(mate_d) > 1 else np.nan,
    })

f360 = pressures.apply(frame_360_features, axis=1)
pressures = pd.concat([pressures, f360], axis=1)
print('Cobertura de freeze frame 360: %.1f%% das pressoes'
      % (100 * f360['n_teammates_10m'].notna().mean()))
print(f360.describe())

### 4.6 Features de Voronoi — Fase 1 (PDF Apêndice A.1)

Diagrama de Voronoi sobre os jogadores visíveis, com células clipadas ao campo 120×80. O portador da bola é o adversário mais próximo da bola; o pressionante é o companheiro mais próximo.

- `voronoi_area_carrier`: área da célula do portador.
- `voronoi_area_presser`: área da célula do pressionante.
- `voronoi_n_opp_neighbors`: nº de adversários com célula adjacente à do portador.

*Observação:* o cálculo dos diagramas pode levar alguns minutos.

In [ ]:
FIELD = Polygon([(0, 0), (FIELD_X, 0), (FIELD_X, FIELD_Y), (0, FIELD_Y)])
VOR_KEYS = ['voronoi_area_carrier', 'voronoi_area_presser', 'voronoi_n_opp_neighbors']

def voronoi_features(row):
    nan_out = pd.Series({k: np.nan for k in VOR_KEYS})
    fr = frames.get(row['id'])
    if fr is None or pd.isna(row['x']):
        return nan_out
    players = fr['freeze_frame']
    if len(players) < 4:
        return nan_out
    locs = [(float(p['location'][0]), float(p['location'][1])) for p in players]
    is_mate = [bool(p.get('teammate')) for p in players]
    try:
        diagram = voronoi_diagram(MultiPoint(locs), envelope=FIELD, edges=False)
        cells = [g.intersection(FIELD) for g in diagram.geoms]
    except Exception:
        return nan_out
    cell_of = [None] * len(players)
    for cell in cells:
        for i, (lx, ly) in enumerate(locs):
            if cell_of[i] is None and cell.covers(Point(lx, ly)):
                cell_of[i] = cell
    bx, by = row['x'], row['y']
    dist = [np.hypot(lx - bx, ly - by) for lx, ly in locs]
    opp = [i for i in range(len(players)) if not is_mate[i]]
    mate = [i for i in range(len(players)) if is_mate[i]]
    if not opp or not mate:
        return nan_out
    ci = min(opp, key=lambda i: dist[i])
    pi = min(mate, key=lambda i: dist[i])
    carrier, presser = cell_of[ci], cell_of[pi]
    if carrier is None or presser is None or carrier.is_empty:
        return nan_out
    neigh = 0
    for i in opp:
        if i == ci:
            continue
        c = cell_of[i]
        if c is not None and not c.is_empty and carrier.touches(c):
            neigh += 1
    return pd.Series({
        'voronoi_area_carrier': float(carrier.area),
        'voronoi_area_presser': float(presser.area),
        'voronoi_n_opp_neighbors': int(neigh),
    })

print('Calculando diagramas de Voronoi (pode levar alguns minutos)...')
vor = pressures.apply(voronoi_features, axis=1)
pressures = pd.concat([pressures, vor], axis=1)
print('Voronoi calculado para %.1f%% das pressoes'
      % (100 * vor['voronoi_area_carrier'].notna().mean()))
print(vor.describe())

### 4.7 Dataset final

Monta-se `df` com uma linha por pressão: a resposta `recovered`, o time pressionante e todas as features. Categorias raras de `trigger_type` e `action_under_pressure` são agrupadas em `Other`.

In [ ]:
TRIG_KEEP = ['Pass', 'Carry', 'Ball Receipt*', 'Dribble']
AUP_KEEP = ['Pass', 'Carry', 'Ball Receipt*', 'Dribble']

def collapse(series, keep):
    return series.where(series.isin(keep), other='Other').fillna('Other')

pressures['trigger_type'] = collapse(pressures['trigger_type'], TRIG_KEEP)
pressures['action_under_pressure'] = collapse(pressures['action_under_pressure'], AUP_KEEP)

CONT = ['dist_to_goal', 'duration', 'minute', 'score_diff',
        'n_teammates_10m', 'n_adversaries_5m', 'n_opponents_10m',
        'nearest_teammate_dist', 'nearest_opponent_dist',
        'numerical_superiority', 'pressing_compactness',
        'voronoi_area_carrier', 'voronoi_area_presser', 'voronoi_n_opp_neighbors']
CAT = ['zone', 'trigger_type', 'action_under_pressure',
       'is_counterpress', 'had_tackle', 'had_block', 'had_foul']
BIN = ['is_counterpress', 'had_tackle', 'had_block', 'had_foul']

df = pressures[['recovered', 'team_name'] + CONT + CAT].copy()
for c in BIN:
    df[c] = df[c].fillna(0).astype(int)

print('Dataset final:', df.shape)
print('Taxa de recuperacao:', round(100 * df['recovered'].mean(), 1), '%')
print('Valores ausentes por coluna:')
print(df.isna().sum()[df.isna().sum() > 0])
df.head()

## 5. Avaliação Estatística do Input (Testes de Hipótese)

Esta é a etapa central: cada feature candidata é submetida a testes formais antes de compor o modelo.

### 5.1 Triagem univariada (qui-quadrado / Mann-Whitney) — `TEST_LEVEL >= 1`

- **Features contínuas:** teste de Mann-Whitney U (não-paramétrico) e teste t de Welch comparando os grupos `recovered = 0` e `recovered = 1`.
- **Features categóricas:** teste qui-quadrado de independência (feature × `recovered`) com Cramér's V como tamanho de efeito.
- Correção para múltiplos testes: Bonferroni e Benjamini-Hochberg (FDR).

In [ ]:
records = []
for c in CONT:
    g0 = df.loc[df['recovered'] == 0, c].dropna()
    g1 = df.loc[df['recovered'] == 1, c].dropna()
    u_stat, p_mw = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    t_stat, p_t = stats.ttest_ind(g0, g1, equal_var=False)
    records.append({'feature': c, 'tipo': 'continua', 'teste': 'Mann-Whitney U',
                    'estatistica': u_stat, 'p_valor': p_mw, 'p_t_welch': p_t,
                    'media_rec0': g0.mean(), 'media_rec1': g1.mean(), 'efeito': np.nan})

for c in CAT:
    table = pd.crosstab(df[c].astype(str), df['recovered'])
    chi2, p_chi, dof, _ = stats.chi2_contingency(table)
    n = table.values.sum()
    cramer = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    records.append({'feature': c, 'tipo': 'categorica', 'teste': 'Qui-quadrado',
                    'estatistica': chi2, 'p_valor': p_chi, 'p_t_welch': np.nan,
                    'media_rec0': np.nan, 'media_rec1': np.nan, 'efeito': cramer})

df_univariate = pd.DataFrame(records)
p_safe = df_univariate['p_valor'].fillna(1.0)
df_univariate['p_bonferroni'] = multipletests(p_safe, method='bonferroni')[1]
df_univariate['p_bh'] = multipletests(p_safe, method='fdr_bh')[1]
df_univariate['significativa'] = df_univariate['p_bh'] < ALPHA
print('Features significativas (BH, alpha=%.2f): %d de %d'
      % (ALPHA, df_univariate['significativa'].sum(), len(df_univariate)))
df_univariate.sort_values('p_valor').round(5)

### 5.2 Multicolinearidade — VIF — `TEST_LEVEL >= 2`

O *Variance Inflation Factor* é calculado sobre as features contínuas padronizadas. VIF ≥ 5 indica multicolinearidade relevante — a feature torna-se candidata a substituição (ver Apêndice A.4 do PDF).

In [ ]:
if TEST_LEVEL >= 2:
    Xc = df[CONT].dropna()
    Xz = (Xc - Xc.mean()) / Xc.std()
    Xz = sm.add_constant(Xz)
    vif_vals = [variance_inflation_factor(Xz.values, i + 1) for i in range(len(CONT))]
    df_vif = pd.DataFrame({'feature': CONT, 'VIF': vif_vals})
    df_vif['multicolinear'] = df_vif['VIF'] >= 5
    print('VIF calculado sobre', len(Xc), 'pressoes com features 360/Voronoi completas.')
    display(df_vif.sort_values('VIF', ascending=False).round(3))
else:
    df_vif = None
    print('TEST_LEVEL < 2: VIF nao executado.')

### 5.3 Significância no modelo logístico — Wald — `TEST_LEVEL >= 2`

Ajusta-se a regressão logística com **efeitos fixos de time** (`C(team_name)`). O teste de Wald avalia a significância de cada coeficiente. São reportados também os *odds ratios*, o Pseudo R² de McFadden e o AIC.

In [ ]:
def build_formula(cont, cat, team=True):
    parts = list(cont) + ['C(%s)' % c for c in cat]
    if team:
        parts.append('C(team_name)')
    return 'recovered ~ ' + ' + '.join(parts)

if TEST_LEVEL >= 2:
    model_df = df.dropna(subset=CONT + CAT + ['recovered', 'team_name']).copy()
    print('Ajustando modelo logistico em', len(model_df), 'pressoes...')
    logit_full = smf.logit(build_formula(CONT, CAT), data=model_df).fit(disp=False, maxiter=200)
    print(logit_full.summary())
    or_ci = np.exp(logit_full.conf_int())
    or_ci.columns = ['or_ic_inf', 'or_ic_sup']
    df_wald = pd.DataFrame({
        'termo': logit_full.params.index,
        'coef': logit_full.params.values,
        'odds_ratio': np.exp(logit_full.params.values),
        'or_ic_inf': or_ci['or_ic_inf'].values,
        'or_ic_sup': or_ci['or_ic_sup'].values,
        'p_wald': logit_full.pvalues.values})
    print('Pseudo R2 (McFadden): %.4f | AIC: %.1f' % (logit_full.prsquared, logit_full.aic))
    display(df_wald[~df_wald['termo'].str.contains('team_name')].round(4))
else:
    logit_full = None
    df_wald = None
    print('TEST_LEVEL < 2: modelo logistico / Wald nao executado.')

### 5.4 Teste da razão de verossimilhança — LRT — `TEST_LEVEL >= 2`

Para cada feature, ajusta-se um modelo reduzido (sem a feature) e compara-se com o modelo completo via estatística `2 · (logL_completo − logL_reduzido)`, distribuída como χ². Reporta-se também a variação de AIC (`delta_AIC > 0` indica que incluir a feature reduz o AIC).

In [ ]:
if TEST_LEVEL >= 2:
    llf_full = logit_full.llf
    aic_full = logit_full.aic
    k_full = len(logit_full.params)
    lrt_records = []
    for c in CONT + CAT:
        red_cont = [x for x in CONT if x != c]
        red_cat = [x for x in CAT if x != c]
        red = smf.logit(build_formula(red_cont, red_cat), data=model_df).fit(disp=False, maxiter=200)
        lr = 2 * (llf_full - red.llf)
        ddof = max(k_full - len(red.params), 1)
        lrt_records.append({'feature': c, 'LR_stat': lr, 'df': ddof,
                            'p_lrt': stats.chi2.sf(lr, ddof),
                            'delta_AIC': red.aic - aic_full})
    df_lrt = pd.DataFrame(lrt_records)
    df_lrt['reduz_AIC'] = df_lrt['delta_AIC'] > 0
    display(df_lrt.sort_values('p_lrt').round(4))
else:
    df_lrt = None
    print('TEST_LEVEL < 2: LRT nao executado.')

### 5.5 Teste de permutação — `TEST_LEVEL >= 3`

Para features com significância limítrofe (`0.01 <= p_BH <= 0.10`), o rótulo `recovered` é permutado 1000× para construir uma distribuição nula empírica da estatística de associação (χ² para categóricas, |t de Welch| para contínuas). O p-valor empírico confirma ou descarta a associação.

In [ ]:
PERM_N = 1000

def permutation_pvalue(frame, feature, is_cat, n_perm=PERM_N, seed=42):
    rng = np.random.default_rng(seed)
    sub = frame[[feature, 'recovered']].dropna()
    y = sub['recovered'].to_numpy()
    if is_cat:
        x = sub[feature].astype(str).to_numpy()
        obs = stats.chi2_contingency(pd.crosstab(x, y))[0]
        count = 0
        for _ in range(n_perm):
            yp = rng.permutation(y)
            count += stats.chi2_contingency(pd.crosstab(x, yp))[0] >= obs
    else:
        x = sub[feature].to_numpy()
        obs = abs(stats.ttest_ind(x[y == 0], x[y == 1], equal_var=False)[0])
        count = 0
        for _ in range(n_perm):
            yp = rng.permutation(y)
            count += abs(stats.ttest_ind(x[yp == 0], x[yp == 1], equal_var=False)[0]) >= obs
    return (count + 1) / (n_perm + 1)

if TEST_LEVEL >= 3:
    border = df_univariate[(df_univariate['p_bh'] >= 0.01) & (df_univariate['p_bh'] <= 0.10)]
    if len(border) == 0:
        print('Nenhuma feature com significancia limitrofe (0.01 <= p_BH <= 0.10).')
        df_perm = pd.DataFrame(columns=['feature', 'p_permutacao'])
    else:
        perm_records = []
        for _, r in border.iterrows():
            p_emp = permutation_pvalue(df, r['feature'], r['tipo'] == 'categorica')
            perm_records.append({'feature': r['feature'], 'p_permutacao': p_emp})
            print('  %s: p_permutacao = %.4f' % (r['feature'], p_emp))
        df_perm = pd.DataFrame(perm_records)
else:
    df_perm = None
    print('TEST_LEVEL < 3: teste de permutacao nao executado.')

### 5.6 Tabelas de análise por variável (`variable_analysis/`)

Cada feature candidata tem sua análise completa registrada em arquivos CSV dentro de `variable_analysis/`, prontos para alimentar gráficos posteriores:

- **`summary.csv`** — tabela mestra, uma linha por variável, com todos os campos escalares: identificação (grupo, referência no PDF, tipo, unidade), distribuição/qualidade dos dados, associação univariada (p-valores, `-log10(p)`, tamanho de efeito), multicolinearidade (VIF, correlação máxima), significância no modelo (coeficiente, odds ratio + IC 95%, Wald, LRT, ΔAIC), permutação e a decisão de inclusão.
- **`breakdown_<feature>.csv`** — uma por variável, com a taxa de recuperação desagregada por faixa (decis, para contínuas) ou categoria, com IC 95% de Wilson — formato direto para curvas/barras.

Campos dependentes de `TEST_LEVEL` (VIF, Wald, LRT, permutação) ficam como `NaN` quando o nível não os executa.

In [ ]:
os.makedirs(VAR_DIR, exist_ok=True)

# Metadados por feature: (grupo, referencia PDF, descricao, calculo, unidade)
FEATURE_META = {
    'dist_to_goal': ('Posicional', '2.4.2', 'Distancia euclidiana ao gol adversario', 'sqrt((120-x)^2 + (40-y)^2)', 'metros'),
    'duration': ('Posicional', '2.4.2', 'Duracao do evento de pressao', 'atributo direto do evento', 'segundos'),
    'minute': ('Contexto de jogo', '2.4.3', 'Minuto da partida', 'atributo direto do evento', 'minuto'),
    'score_diff': ('Contexto de jogo', '2.4.3', 'Diferenca de gols (pressionante - adversario)', 'placar corrido ate o timestamp', 'gols'),
    'n_teammates_10m': ('Freeze frame 360', '2.4.1', 'Companheiros em 10m da bola', 'contagem no freeze frame', 'contagem'),
    'n_adversaries_5m': ('Freeze frame 360', '2.4.1', 'Adversarios em 5m da bola', 'contagem no freeze frame', 'contagem'),
    'n_opponents_10m': ('Freeze frame 360', '2.4.1', 'Adversarios em 10m da bola', 'contagem no freeze frame', 'contagem'),
    'nearest_teammate_dist': ('Freeze frame 360', '2.4.1', 'Distancia do companheiro mais proximo a bola', 'minimo das distancias', 'metros'),
    'nearest_opponent_dist': ('Freeze frame 360', '2.4.1', 'Distancia do adversario mais proximo a bola', 'minimo das distancias', 'metros'),
    'numerical_superiority': ('Freeze frame 360', '2.4.1', 'Superioridade numerica local', 'n_teammates_10m - n_opponents_10m', 'contagem'),
    'pressing_compactness': ('Freeze frame 360', '2.4.1', 'Compacidade do bloco de pressing', 'desvio padrao das distancias dos companheiros a bola', 'metros'),
    'voronoi_area_carrier': ('Voronoi', 'A.1', 'Area da celula de Voronoi do portador', 'diagrama de Voronoi clipado ao campo', 'm2'),
    'voronoi_area_presser': ('Voronoi', 'A.1', 'Area da celula de Voronoi do pressionante', 'diagrama de Voronoi clipado ao campo', 'm2'),
    'voronoi_n_opp_neighbors': ('Voronoi', 'A.1', 'Adversarios com celula adjacente a do portador', 'adjacencia no diagrama de Voronoi', 'contagem'),
    'zone': ('Posicional', '2.4.2', 'Terco do campo', 'Defensivo/Medio/Atacante por coordenada x', 'categorica'),
    'trigger_type': ('Tatica', '2.4.4', 'Evento do adversario antes da pressao', 'merge_asof backward 5s', 'categorica'),
    'action_under_pressure': ('Tatica', '2.4.4', 'Acao do adversario sob pressao', 'merge_asof nearest 2s', 'categorica'),
    'is_counterpress': ('Tatica', '2.4.4', 'Pressao ate 5s apos perda do proprio time', 'merge_asof backward 5s', 'binaria'),
    'had_tackle': ('Acoes defensivas concorrentes', '2.4.5', 'Desarme do time pressionante em ate 3s', 'merge_asof forward 3s', 'binaria'),
    'had_block': ('Acoes defensivas concorrentes', '2.4.5', 'Bloqueio do time pressionante em ate 3s', 'merge_asof forward 3s', 'binaria'),
    'had_foul': ('Acoes defensivas concorrentes', '2.4.5', 'Falta cometida pelo time pressionante em ate 3s', 'merge_asof forward 3s', 'binaria'),
}

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

def cohens_d(a, b):
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    sp = np.sqrt(((na - 1) * sa * sa + (nb - 1) * sb * sb) / (na + nb - 2))
    return (b.mean() - a.mean()) / sp if sp > 0 else np.nan

def breakdown_continuous(feature, n_bins=10):
    sub = df[[feature, 'recovered']].dropna()
    try:
        bins = pd.qcut(sub[feature], q=n_bins, duplicates='drop')
    except Exception:
        bins = pd.cut(sub[feature], bins=min(n_bins, max(sub[feature].nunique(), 2)))
    rows = []
    for b, g in sub.groupby(bins, observed=True):
        n = len(g)
        k = int(g['recovered'].sum())
        lo, hi = wilson_ci(k, n)
        rows.append({'feature': feature, 'faixa': str(b), 'valor_representativo': g[feature].mean(),
                     'n': n, 'n_recuperados': k, 'taxa_recuperacao': k / n if n else np.nan,
                     'ic_inf': lo, 'ic_sup': hi})
    return pd.DataFrame(rows)

def breakdown_categorical(feature):
    sub = df[[feature, 'recovered']].copy()
    sub[feature] = sub[feature].astype(str)
    rows = []
    for i, (cat, g) in enumerate(sub.groupby(feature, observed=True)):
        n = len(g)
        k = int(g['recovered'].sum())
        lo, hi = wilson_ci(k, n)
        rows.append({'feature': feature, 'faixa': cat, 'valor_representativo': i,
                     'n': n, 'n_recuperados': k, 'taxa_recuperacao': k / n if n else np.nan,
                     'ic_inf': lo, 'ic_sup': hi})
    return pd.DataFrame(rows)

print('Funcoes e metadados de analise por variavel carregados.')

In [ ]:
# Lookups dependentes do TEST_LEVEL
corr_matrix = df[CONT].corr().abs()
for _col in corr_matrix.columns:
    corr_matrix.loc[_col, _col] = np.nan

wald_lookup = {}
if logit_full is not None:
    ci = np.exp(logit_full.conf_int())
    ci.columns = ['or_lo', 'or_hi']
    for term in logit_full.params.index:
        wald_lookup[term] = {'coef': logit_full.params[term], 'or': np.exp(logit_full.params[term]),
                             'or_lo': ci.loc[term, 'or_lo'], 'or_hi': ci.loc[term, 'or_hi'],
                             'p': logit_full.pvalues[term]}

def wald_for_feature(feature, is_cat):
    if not wald_lookup:
        return []
    if is_cat:
        return [t for t in wald_lookup if t.startswith('C(%s)[' % feature)]
    return [feature] if feature in wald_lookup else []

def analisar_variavel(feature):
    is_cat = feature in CAT
    grupo, pdf_ref, descricao, calculo, unidade = FEATURE_META[feature]
    s = df[feature]
    n_obs = len(s)
    n_valid = int(s.notna().sum())
    n_miss = n_obs - n_valid
    row = {'feature': feature, 'grupo': grupo, 'pdf_referencia': pdf_ref,
           'tipo': 'binaria' if feature in BIN else ('categorica' if is_cat else 'continua'),
           'descricao': descricao, 'calculo': calculo, 'unidade': unidade,
           'n_obs': n_obs, 'n_validos': n_valid, 'n_missing': n_miss,
           'pct_missing': round(100 * n_miss / n_obs, 3),
           'taxa_recuperacao_global': round(df['recovered'].mean(), 4)}
    # Bloco B - distribuicao
    if is_cat:
        vc = s.astype(str).value_counts()
        row.update({'n_categorias': int(vc.size), 'categorias': '|'.join(map(str, vc.index)),
                    'categoria_modal': str(vc.index[0]), 'freq_modal': int(vc.iloc[0]),
                    'categoria_menos_frequente': str(vc.index[-1]), 'freq_minima': int(vc.iloc[-1])})
    else:
        v = s.dropna()
        q1, q3 = v.quantile(0.25), v.quantile(0.75)
        iqr = q3 - q1
        n_out = int(((v < q1 - 1.5 * iqr) | (v > q3 + 1.5 * iqr)).sum())
        row.update({'min': v.min(), 'max': v.max(), 'media': v.mean(), 'mediana': v.median(),
                    'desvio_padrao': v.std(), 'p25': q1, 'p75': q3, 'iqr': iqr,
                    'assimetria': v.skew(), 'curtose': v.kurt(), 'n_outliers': n_out})
    # Bloco C - associacao univariada
    u = df_univariate[df_univariate['feature'] == feature].iloc[0]
    p_raw = u['p_valor']
    row.update({'teste_univariado': u['teste'], 'estatistica_univariada': u['estatistica'],
                'p_valor': p_raw, 'p_bonferroni': u['p_bonferroni'], 'p_bh': u['p_bh'],
                'neg_log10_p': -np.log10(p_raw) if pd.notna(p_raw) and p_raw > 0 else np.nan,
                'significativa_univariada': bool(u['significativa'])})
    if is_cat:
        rates = df.groupby(df[feature].astype(str))['recovered'].mean()
        row.update({'tamanho_efeito': u['efeito'], 'tipo_efeito': 'Cramers V',
                    'media_rec0': np.nan, 'media_rec1': np.nan,
                    'mediana_rec0': np.nan, 'mediana_rec1': np.nan,
                    'diff_medias': np.nan, 'diff_padronizada': np.nan,
                    'direcao_efeito': 'maior taxa: %s' % rates.idxmax()})
    else:
        g0 = df.loc[df['recovered'] == 0, feature].dropna()
        g1 = df.loc[df['recovered'] == 1, feature].dropna()
        d = cohens_d(g0, g1)
        row.update({'tamanho_efeito': d, 'tipo_efeito': 'Cohen d',
                    'media_rec0': g0.mean(), 'media_rec1': g1.mean(),
                    'mediana_rec0': g0.median(), 'mediana_rec1': g1.median(),
                    'diff_medias': g1.mean() - g0.mean(), 'diff_padronizada': d,
                    'direcao_efeito': 'recupera mais com valor ' + ('alto' if g1.mean() >= g0.mean() else 'baixo')})
    # Bloco D - multicolinearidade
    if df_vif is not None and not is_cat:
        vr = df_vif[df_vif['feature'] == feature]
        row['VIF'] = float(vr['VIF'].iloc[0]) if len(vr) else np.nan
        row['multicolinear'] = bool(vr['multicolinear'].iloc[0]) if len(vr) else False
    else:
        row['VIF'] = np.nan
        row['multicolinear'] = False
    if not is_cat and feature in corr_matrix.columns:
        cm = corr_matrix[feature]
        row['corr_max_abs'] = cm.max()
        row['feature_mais_correlacionada'] = cm.idxmax() if cm.notna().any() else None
    else:
        row['corr_max_abs'] = np.nan
        row['feature_mais_correlacionada'] = None
    row['candidata_substituicao'] = bool((pd.notna(row['corr_max_abs']) and row['corr_max_abs'] > 0.8)
                                         or row['multicolinear'])
    # Bloco E - Wald
    terms = wald_for_feature(feature, is_cat)
    if terms:
        best = min(terms, key=lambda t: wald_lookup[t]['p'])
        w = wald_lookup[best]
        row.update({'coef_logit': w['coef'], 'odds_ratio': w['or'],
                    'or_ic_inf': w['or_lo'], 'or_ic_sup': w['or_hi'], 'p_wald': w['p']})
    else:
        row.update({'coef_logit': np.nan, 'odds_ratio': np.nan,
                    'or_ic_inf': np.nan, 'or_ic_sup': np.nan, 'p_wald': np.nan})
    # LRT
    if df_lrt is not None:
        lr = df_lrt[df_lrt['feature'] == feature]
        if len(lr):
            row.update({'LR_stat': lr['LR_stat'].iloc[0], 'df_lrt': lr['df'].iloc[0],
                        'p_lrt': lr['p_lrt'].iloc[0], 'delta_AIC': lr['delta_AIC'].iloc[0],
                        'reduz_AIC': bool(lr['reduz_AIC'].iloc[0])})
        else:
            row.update({'LR_stat': np.nan, 'df_lrt': np.nan, 'p_lrt': np.nan,
                        'delta_AIC': np.nan, 'reduz_AIC': False})
    else:
        row.update({'LR_stat': np.nan, 'df_lrt': np.nan, 'p_lrt': np.nan,
                    'delta_AIC': np.nan, 'reduz_AIC': False})
    row['significativa_modelo'] = bool(pd.notna(row['p_lrt']) and row['p_lrt'] < ALPHA)
    # Bloco F - permutacao
    if df_perm is not None and len(df_perm) and feature in set(df_perm['feature']):
        row['p_permutacao'] = float(df_perm.loc[df_perm['feature'] == feature, 'p_permutacao'].iloc[0])
        row['n_permutacoes'] = PERM_N
    else:
        row['p_permutacao'] = np.nan
        row['n_permutacoes'] = np.nan
    # Bloco G - decisao
    if not row['significativa_univariada']:
        dec, mot = 'EXCLUIR', 'sem associacao univariada (p_BH >= alpha)'
    elif row['multicolinear']:
        dec, mot = 'REVISAR', 'multicolinearidade (VIF >= 5)'
    elif df_lrt is not None and pd.notna(row['p_lrt']) and row['p_lrt'] >= ALPHA:
        dec, mot = 'EXCLUIR', 'nao significativa no LRT'
    else:
        dec, mot = 'INCLUIR', 'aprovada nos testes executados'
    row['decisao'] = dec
    row['motivo_decisao'] = mot
    row['test_level'] = TEST_LEVEL
    row['data_analise'] = datetime.now().isoformat(timespec='seconds')
    # tabela de breakdown por variavel
    bd = breakdown_categorical(feature) if is_cat else breakdown_continuous(feature)
    bd.to_csv(os.path.join(VAR_DIR, 'breakdown_%s.csv' % feature), index=False)
    return row

df_variable_analysis = pd.DataFrame([analisar_variavel(f) for f in CONT + CAT])
df_variable_analysis.to_csv(os.path.join(VAR_DIR, 'summary.csv'), index=False)
print('Tabela mestra salva em', os.path.join(VAR_DIR, 'summary.csv'), '| shape:', df_variable_analysis.shape)
print('Arquivos de breakdown gerados:', len(CONT + CAT), '(um por variavel)')
display(df_variable_analysis)

## 6. Tabela Consolidada e Decisão de Inclusão

Visão resumida da tabela mestra `variable_analysis/summary.csv` (montada na Seção 5.6), incluindo as features rejeitadas — a tabela de transparência exigida pela Seção A.5 do PDF. A decisão final segue a regra:

- **EXCLUIR** se não houver associação univariada (BH);
- **REVISAR** se VIF ≥ 5 (candidata a substituição por multicolinearidade);
- **EXCLUIR** se não significativa no LRT (`TEST_LEVEL >= 2`);
- **INCLUIR** caso contrário.

In [ ]:
cols_view = ['feature', 'grupo', 'tipo', 'p_valor', 'p_bh', 'significativa_univariada',
             'tamanho_efeito', 'VIF', 'p_lrt', 'delta_AIC', 'p_permutacao', 'decisao']
consol = df_variable_analysis[cols_view].copy()
display(consol.sort_values('p_valor').round(4))
print()
print('Resumo das decisoes:')
print(df_variable_analysis['decisao'].value_counts())
print()
print('Arquivos gerados em', VAR_DIR + '/:')
print(' - summary.csv (', len(df_variable_analysis), 'variaveis )')
print(' - breakdown_<feature>.csv (', len(CONT + CAT), 'arquivos )')

## 7. Conclusões

A tabela mestra `variable_analysis/summary.csv` documenta, para cada feature candidata, o resultado de todos os testes de hipótese executados e a decisão de inclusão — exatamente a tabela de transparência exigida pela Seção A.5 do PDF. As tabelas `breakdown_<feature>.csv` guardam a taxa de recuperação por faixa/categoria, prontas para gerar gráficos (curva de recuperação, ranking de importância, volcano plot, forest plot de odds ratios).

**Interpretação:**

- Features marcadas como **INCLUIR** têm associação estatística confirmada com a recuperação de bola e devem compor a especificação final do modelo logístico.
- Features marcadas como **REVISAR** apresentam multicolinearidade (VIF ≥ 5); são candidatas a substituição — por exemplo, `numerical_superiority` versus `n_teammates_10m`/`n_opponents_10m`, ou as features de Voronoi versus as contagens da Seção 2.4.1.
- Features marcadas como **EXCLUIR** não justificam empiricamente sua inclusão.

**Próximos passos (etapas futuras):**

1. Gerar os gráficos a partir de `variable_analysis/` (ranking de importância, forest plot, curvas de recuperação).
2. Ajuste do modelo logístico final com efeitos fixos de time, usando apenas as features aprovadas (PDF 2.5).
3. Análise de resíduos por jogador para a qualidade individual de pressing (PDF 2.6).
4. Análise de contrapressão e de triggers táticos (PDF 2.7–2.8).
5. Comparação dos modelos M1/M2/M3 com e sem as features de Voronoi (PDF Apêndice A.4).

Para executar a bateria completa de testes, ajuste `TEST_LEVEL = 2` ou `TEST_LEVEL = 3` na célula de configuração e reexecute o notebook.